# DatensEE Quickstart — Colab / Notebook

Export Earth Engine imagery at scale via Cloud Dataflow, directly from a notebook.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/michaelfdewitt/datensee/blob/master/notebooks/datensee_quickstart.ipynb)

## 1. Install DatensEE

In [ ]:
!pip install -q "datensee[validation]"

### Java runtime (Colab, local runner only)

The local runner executes a small Java pipeline, which needs Java 21+. Colab ships an older JVM, so install 21 if needed. Dataflow mode does not need any local Java and can skip this.

In [ ]:
import os, re, subprocess

def _java_major():
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except FileNotFoundError:
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0

if _java_major() < 21:
    !apt-get -qq install -y openjdk-21-jre-headless > /dev/null
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
    os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0])


## 2. Authenticate

In Colab, this triggers the interactive Google auth flow.
Outside Colab, ensure ADC is configured (`gcloud auth application-default login`).

In [ ]:
import datensee
from datensee import notebook

notebook.ensure_auth()

## 3. Define the export

Use the built-in demo (Landsat 9 NDVI over SF Bay Area) or define your own
expression with `ee.serializer.encode(image, for_cloud_api=True)`.

In [ ]:
from datensee.api import demo_expression, demo_region

# For a custom expression, author it with your own ee client, e.g.:
#   import ee
#   ee.Initialize(project="your-project")
#   image = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
#            .filterDate('2023-06-01', '2023-09-01').median()
#            .normalizedDifference(['SR_B5', 'SR_B4']))
#   expression = image        # datensee serializes it for you
#   region = ee.Geometry.Rectangle([-122.5, 37.75, -122.25, 38.0])

expression = demo_expression()
region = demo_region()

PROJECT = "your-gcp-project"  # <-- change this


## 4. Tile the region

In [ ]:
grid = datensee.tile(region, scale=30.0)
print(f"{len(grid.tiles)} tiles at {grid.pixel_size:g} {grid.crs} units/px")

## 5. Review export config

In [ ]:
# Review the export before running it. A dry run validates inputs and renders
# the summary (tiles, pixel size, estimated cost); nothing is submitted.
datensee.export(
    ee_expression=expression,
    region=region,
    project=PROJECT,
    output="./datensee-output",
    scale=30.0,
    runner="local",
    dry_run=True,
    confirm_callback=notebook.display_export_summary,
)


## 6. Run the export

For small regions, use `runner="local"` with a local output directory.
For large regions, use `runner="dataflow"` with a GCS output path.

In [ ]:
result = datensee.export(
    ee_expression=expression,
    region=region,
    project=PROJECT,
    output="./datensee-output",
    runner="local",
)

print(f"Tiles OK: {result.tiles_ok}")
print(f"Duration: {result.duration_seconds:.1f}s")

## 7. Monitor a Dataflow job (optional)

If you used `runner="dataflow"`, poll the job with an HTML status display.

In [ ]:
# Uncomment to monitor a Dataflow job:
# final_state = notebook.display_job_progress(
#     result.job_id,
#     project=PROJECT,
#     region="us-central1",
# )

## 8. View the output

Output files are plain Cloud Optimized GeoTIFFs — open them directly in QGIS or any
other GIS tool. To peek at one from Python (needs `rasterio` and `matplotlib`):

```python
import rasterio, matplotlib.pyplot as plt
with rasterio.open("./datensee-output/tile_r0000_c0000.tif") as src:
    plt.imshow(src.read(1), cmap="viridis")
```

## 9. Validate output (optional)

In [ ]:
# from datensee.pixel.validation import validate_output
# report = validate_output("./datensee-output", result.config)
# print(report.render())